# 15 · Provider failure drill
Level L2 · Part 2 · professional · ~35 min · builds on 11, 14

## Goal
Run Scout's streaming client against a fake provider that fails in six ways (a 429 without
`retry-after`, a spend-cap 429, a 529 overloaded, an error event mid-stream, a mid-stream disconnect and
truncated tool JSON), and print a before/during/after table showing Scout handled each one correctly.

## The idea
Lesson 14 handled errors that arrive as a status code before any reply. Streaming (lesson 11) adds a
worse kind: the server says 200, sends half the answer, then fails. The SDK cannot retry that for you.
So we sort failures into a **failure taxonomy**: four groups, each with one correct action. Then we
run a **chaos drill**: break the provider on purpose, check Scout before, during and after the
failure, and read the table.

Analogy: a fire drill. You do not wait for a real fire to learn whether the exits work. You set off
the alarm on a calm day, watch what people do, then check everyone is back at their desks.

## Picture
```mermaid
flowchart LR
    F["failure"] --> T{"which group?"}
    T -->|"429, 529, 5xx"| A["back off, retry"]
    T -->|"spend cap, 400"| B["stop, alert a human"]
    T -->|"broken stream"| C["drop partial text, retry whole call"]
    T -->|"truncated tool JSON"| D["retry with more max_tokens"]
```

## Step by step
### 1. Setup from lesson 14
The three pieces of lesson 14's retry loop, unchanged except for a smaller `base` so the drill runs
fast. `retry_decision` already treats 529 like any 5xx: temporary.

In [1]:
import json, random, time

class GiveUp(Exception):
    pass

def backoff_delay(attempt, base=0.2, cap=2.0, rng=random):
    return rng.uniform(0, min(cap, base * 2 ** attempt))

def retry_decision(status, body) -> tuple[bool, str]:
    if status == 200:
        return False, "ok"
    if status is None:
        return True, "timeout or network error"
    details = (body.get("error") or {}).get("details") or {}
    if isinstance(details, dict) and details.get("error_code") == "enforced_spend_limit_reached":
        return False, "spend cap reached: wait for next month"
    if status in (408, 429) or status >= 500:
        return True, f"{status}: temporary"
    return False, f"{status}: fix the request"

print(retry_decision(529, {"error": {"type": "overloaded_error"}}))

(True, '529: temporary')


### 2. A provider you can break
The stub speaks Anthropic's streaming format (server-sent events, lesson 11). `POST /fault` switches
on one failure for the next `times` requests; `GET /` reports the fault and the hit count. Read
`stream_events`: that is where the three mid-stream failures happen.

We write the server file in three parts. The first part holds the state, the error and SSE helpers,
and the event generator. It uses only the standard library.

In [2]:
%%writefile ../apps/15/stub_server.py
"""Scout's chaos stub: an Anthropic-shaped streaming provider with six injectable failures."""
import json, sys
from http.server import BaseHTTPRequestHandler, ThreadingHTTPServer

STATE = {"fault": "none", "left": 0, "hits": 0}     # current fault, failures left, requests seen
TEXT = ["Perseverance ", "landed on Mars ", "in February 2021."]
TOOL_JSON = ['{"query": ', '"Perseverance ', 'landing date"}']

def err(kind, msg, **details):
    e = {"type": kind, "message": msg} | ({"details": details} if details else {})
    return {"type": "error", "error": e, "request_id": "req_stub"}

def sse(event, data):
    return f"event: {event}\ndata: {json.dumps(data)}\n\n".encode()

def stream_events(fault, max_tokens):
    msg = {"id": "msg_stub", "type": "message", "role": "assistant", "model": "stub", "content": [],
           "stop_reason": None, "stop_sequence": None, "usage": {"input_tokens": 20, "output_tokens": 0}}
    yield sse("message_start", {"type": "message_start", "message": msg})
    tool = fault == "truncated_tool"
    block = ({"type": "tool_use", "id": "toolu_stub", "name": "search_notes", "input": {}} if tool
             else {"type": "text", "text": ""})
    yield sse("content_block_start", {"type": "content_block_start", "index": 0, "content_block": block})
    cut = tool and max_tokens < 100                  # too few tokens: the JSON stops half way
    for i, piece in enumerate((TOOL_JSON if tool else TEXT)[: 2 if cut else 3]):
        delta = {"type": "input_json_delta", "partial_json": piece} if tool else {"type": "text_delta", "text": piece}
        yield sse("content_block_delta", {"type": "content_block_delta", "index": 0, "delta": delta})
        if i == 1 and fault == "stream_error":       # 200 already sent, then an error event
            yield sse("error", err("overloaded_error", "Overloaded"))
            return
        if i == 1 and fault == "disconnect":         # 200 already sent, then the line goes dead
            return
    yield sse("content_block_stop", {"type": "content_block_stop", "index": 0})
    stop = "max_tokens" if cut else ("tool_use" if tool else "end_turn")
    yield sse("message_delta", {"type": "message_delta", "delta": {"stop_reason": stop, "stop_sequence": None},
                                "usage": {"output_tokens": 12}})
    yield sse("message_stop", {"type": "message_stop"})

Overwriting ../apps/15/stub_server.py


Part two is the request handler. A `disconnect` promises 500 more bytes in `content-length` than it
sends, then hangs up: exactly what a dropped connection looks like to the client.

In [3]:
%%writefile -a ../apps/15/stub_server.py

class Handler(BaseHTTPRequestHandler):
    protocol_version = "HTTP/1.1"
    def log_message(self, *args):
        pass

    def send(self, code, data, ctype="application/json", extra=0):
        self.send_response(code)
        self.send_header("content-type", ctype)
        self.send_header("content-length", str(len(data) + extra))   # extra > 0: promise more than we send
        self.end_headers()
        self.wfile.write(data)

    def do_GET(self):
        self.send(200, json.dumps(STATE).encode())

    def do_POST(self):
        req = json.loads(self.rfile.read(int(self.headers.get("content-length", 0))) or b"{}")
        if self.path == "/fault":
            STATE.update(fault=req["fault"], left=req.get("times", 10**6), hits=0)
            return self.send(200, json.dumps(STATE).encode())
        STATE["hits"] += 1
        fault = STATE["fault"] if STATE["left"] > 0 else "none"
        STATE["left"] -= 1
        if fault == "ratelimit":                     # a 429 with no retry-after header
            return self.send(429, json.dumps(err("rate_limit_error", "Rate limit exceeded.")).encode())
        if fault == "spendcap":
            body = err("rate_limit_error", "You have reached your API usage limits. You will regain access "
                       "on 2026-11-01 at 00:00 UTC.", error_code="enforced_spend_limit_reached")
            return self.send(429, json.dumps(body).encode())
        if fault == "overloaded":
            return self.send(529, json.dumps(err("overloaded_error", "Overloaded")).encode())
        cut_off = fault == "disconnect"
        self.send(200, b"".join(stream_events(fault, req.get("max_tokens", 0))), "text/event-stream", 500 * cut_off)
        self.close_connection = cut_off

Appending to ../apps/15/stub_server.py


Part three starts the server and stays quiet when a client hangs up mid-stream.

In [4]:
%%writefile -a ../apps/15/stub_server.py

class QuietServer(ThreadingHTTPServer):
    def handle_error(self, *args):                   # a client hanging up is expected here
        pass

if __name__ == "__main__":
    QuietServer(("127.0.0.1", int(sys.argv[1])), Handler).serve_forever()

Appending to ../apps/15/stub_server.py


Start it on this lesson's port, 8715, and poll until it answers. Two helpers drive the drill: `fault`
breaks the provider, `stub_state` reads it back.

In [5]:
import atexit, socket, subprocess, sys, urllib.request

PORT, BASE = 8715, "http://127.0.0.1:8715"
with socket.socket() as s:
    assert s.connect_ex(("127.0.0.1", PORT)) != 0, f"port {PORT} is busy"
server = subprocess.Popen([sys.executable, "../apps/15/stub_server.py", str(PORT)],
                          stdout=subprocess.DEVNULL, stderr=subprocess.DEVNULL)
atexit.register(server.terminate)

def stub_state():
    return json.load(urllib.request.urlopen(BASE + "/", timeout=1))

def fault(name, times=10**6):
    req = urllib.request.Request(BASE + "/fault", data=json.dumps({"fault": name, "times": times}).encode(),
                                 method="POST")
    return json.load(urllib.request.urlopen(req, timeout=1))

for _ in range(50):
    try:
        print("health:", stub_state())
        break
    except OSError:
        time.sleep(0.1)

health: {'fault': 'none', 'left': 0, 'hits': 0}


### 3. Scout's streaming client
The same streaming idea as lesson 11, now with Anthropic's event names. Scout collects text deltas,
tool-argument deltas (`input_json_delta`, partial JSON strings), the stop reason, and whether the final
`message_stop` arrived. `max_retries=0`: our loop decides about retries, as in lesson 14.
Lesson 14 already used the Anthropic SDK against a stub; we use it here because its streaming
error format is documented (RESEARCH §37).

In [6]:
import anthropic

client = anthropic.Anthropic(api_key="stub", base_url=BASE, max_retries=0)
TOOLS = [{"name": "search_notes", "description": "Search Scout's notes.",
          "input_schema": {"type": "object", "properties": {"query": {"type": "string"}}}}]

def scout_stream(max_tokens=60):
    out = {"text": "", "tool_json": "", "stop_reason": None, "finished": False}
    events = client.messages.create(model="claude-haiku-4-5", max_tokens=max_tokens, tools=TOOLS,
                                    messages=[{"role": "user", "content": "When did Perseverance land?"}],
                                    stream=True)
    for ev in events:
        if ev.type == "content_block_delta" and ev.delta.type == "text_delta":
            out["text"] += ev.delta.text
        elif ev.type == "content_block_delta" and ev.delta.type == "input_json_delta":
            out["tool_json"] += ev.delta.partial_json
        elif ev.type == "message_delta":
            out["stop_reason"] = ev.delta.stop_reason
        elif ev.type == "message_stop":
            out["finished"] = True
    return out

fault("none")
print(scout_stream())

{'text': 'Perseverance landed on Mars in February 2021.', 'tool_json': '', 'stop_reason': 'end_turn', 'finished': True}


### 4. The naive client meets six failures
First, no protection at all: call once, and if a tool was requested, parse its JSON. We record what
each failure looks like to the caller: an exception type, or a result.

In [7]:
FAULTS = ["ratelimit", "spendcap", "overloaded", "stream_error", "disconnect", "truncated_tool"]

def naive_call():
    out = scout_stream()
    if out["tool_json"]:
        out["tool_args"] = json.loads(out["tool_json"])
    return out

lines = []                                  # collect, then print once: one clean block of output
for name in FAULTS:
    fault(name)
    try:
        out = naive_call()
        lines.append(f"{name:15} -> returned {out}")
    except Exception as e:
        lines.append(f"{name:15} -> {type(e).__module__}.{type(e).__name__} (status {getattr(e, 'status_code', None)})")
print("\n".join(lines))

ratelimit       -> anthropic.RateLimitError (status 429)
spendcap        -> anthropic.RateLimitError (status 429)
overloaded      -> anthropic.OverloadedError (status 529)
stream_error    -> anthropic.APIStatusError (status 200)
disconnect      -> httpx2.RemoteProtocolError (status None)
truncated_tool  -> json.decoder.JSONDecodeError (status None)


Six failures, five different exceptions. Three surprises:
- `stream_error` is an `APIStatusError` with **status 200**: the HTTP response was fine, the error came
  inside it as an `event: error`.
- `disconnect` is not an `anthropic` exception at all. It is `httpx2.RemoteProtocolError` from the HTTP
  library underneath (anthropic 1.11), so `except anthropic.APIError` misses it.
- `truncated_tool` got all the way to `json.loads` before failing.

### 5. What the SDK's own retries cover
Does the SDK's default retrying (`max_retries=2`, lesson 14) save us? Try it on the 529 and on the
mid-stream error, each set to fail twice.

In [8]:
sdk = anthropic.Anthropic(api_key="stub", base_url=BASE)            # default max_retries=2
lines = []
for name in ("overloaded", "stream_error"):
    fault(name, times=2)
    try:
        list(sdk.messages.create(model="claude-haiku-4-5", max_tokens=60, stream=True,
                                 messages=[{"role": "user", "content": "When did Perseverance land?"}]))
        lines.append(f"{name:13} -> recovered after {stub_state()['hits']} requests")
    except anthropic.APIStatusError as e:
        lines.append(f"{name:13} -> {type(e).__name__} {e.status_code} after {stub_state()['hits']} request(s)")
print("\n".join(lines))

overloaded    -> recovered after 3 requests
stream_error  -> APIStatusError 200 after 1 request(s)


The SDK retried the 529 and recovered. The mid-stream error came after a 200, when the SDK had already
handed the stream to us, so it was raised after one request. Anthropic's docs say it plainly: an error
after a 200 "doesn't follow these standard mechanisms". Mid-stream failures are your code's job.

### 6. The failure taxonomy
One function sorts any outcome into a group, so every caller agrees. Status errors go through lesson
14's `retry_decision`. A 200 with an error event, or a dead connection, is a **broken stream**. A
stream that ends without `message_stop` is broken too. A `max_tokens` stop with tool JSON is
**truncated**.

In [9]:
import httpx2

def classify(exc=None, out=None) -> tuple[str, str]:
    if isinstance(exc, anthropic.APIStatusError) and exc.status_code == 200:
        return "broken stream", f"error event mid-stream: {exc.body['error']['type']}"
    if isinstance(exc, anthropic.APIStatusError):
        retry, why = retry_decision(exc.status_code, exc.body or {})
        return ("transient" if retry else "fatal"), why
    if isinstance(exc, (anthropic.APIConnectionError, httpx2.TransportError)):
        return "broken stream", f"connection lost: {type(exc).__name__}"
    if not out["finished"]:
        return "broken stream", "stream ended without message_stop"
    if out["stop_reason"] == "max_tokens" and out["tool_json"]:
        return "truncated", "tool JSON cut off by max_tokens"
    return "ok", out["stop_reason"]

print(classify(out={"finished": True, "stop_reason": "max_tokens", "tool_json": '{"query": '}))

('truncated', 'tool JSON cut off by max_tokens')


Run the taxonomy over the six failures. Each row is the group and the reason.

In [10]:
TAXONOMY = {}
for name in FAULTS:
    fault(name)
    try:
        TAXONOMY[name] = classify(out=scout_stream())
    except Exception as e:
        TAXONOMY[name] = classify(exc=e)
print("\n".join(f"{name:15} -> {group:13} | {why}" for name, (group, why) in TAXONOMY.items()))

ratelimit       -> transient     | 429: temporary
spendcap        -> fatal         | spend cap reached: wait for next month
overloaded      -> transient     | 529: temporary
stream_error    -> broken stream | error event mid-stream: overloaded_error
disconnect      -> broken stream | connection lost: RemoteProtocolError
truncated_tool  -> truncated     | tool JSON cut off by max_tokens


### 7. The hardened call
`scout_call` takes any `stream_fn(max_tokens)` and acts on the group. Transient: back off. Fatal:
raise at once. Broken stream: throw away the partial text (it was incomplete, and we never showed it),
then retry the whole call. Truncated: retry with four times the `max_tokens`. Every attempt is logged.

In [11]:
def scout_call(stream_fn, max_attempts=4, max_tokens=60, rng=random.Random(15), log=None):
    log = [] if log is None else log
    for attempt in range(max_attempts):
        out, exc = None, None
        try:
            out = stream_fn(max_tokens)
        except (anthropic.APIError, httpx2.HTTPError) as e:     # anything else is a bug: let it raise
            exc = e
        kind, why = classify(exc, out)
        log.append(f"{kind}: {why}")
        if kind == "ok":
            if out["tool_json"]:
                out["tool_args"] = json.loads(out["tool_json"])
            return out | {"attempts": attempt + 1}
        if kind == "fatal" or attempt == max_attempts - 1:
            raise GiveUp(f"{kind}: {why}")
        if kind == "truncated":
            max_tokens *= 4                           # a bigger budget, no need to wait
        else:
            time.sleep(backoff_delay(attempt, rng=rng))

One failure first: the mid-stream error, set to happen once. The log shows the broken attempt and the
clean retry.

In [12]:
fault("stream_error", times=1)
log = []
out = scout_call(scout_stream, log=log)
print(log)
print(out["text"], "| attempts:", out["attempts"])

['broken stream: error event mid-stream: overloaded_error', 'ok: end_turn']
Perseverance landed on Mars in February 2021. | attempts: 2


### 8. The drill: before, during, after
For each failure: a healthy call **before**, the failure injected **during**, the fault cleared
**after**. Transient faults fail twice, broken streams once, the spend cap and the short tool budget
never clear by themselves. `hits` counts requests the provider saw during the fault.

In [13]:
TIMES = {"ratelimit": 2, "spendcap": 10**6, "overloaded": 2, "stream_error": 1, "disconnect": 1,
         "truncated_tool": 10**6}

def drill(name):
    row = {"failure": name, "group": TAXONOMY[name][0]}
    for phase in ("before", "during", "after"):
        fault(name if phase == "during" else "none", TIMES[name])
        try:
            row[phase] = f"ok in {scout_call(scout_stream)['attempts']}"
        except GiveUp as e:
            row[phase] = "gave up"
        if phase == "during":
            row["hits"] = stub_state()["hits"]
    return row

ROWS = [drill(name) for name in FAULTS]
print(len(ROWS), "drills run")

6 drills run


The table. The correct result is "ok" everywhere except the spend cap during the fault, which must
give up after exactly one request.

In [14]:
print(f"{'failure':15} {'group':14} {'before':11} {'during':11} {'after':11} hits")
for r in ROWS:
    print(f"{r['failure']:15} {r['group']:14} {r['before']:11} {r['during']:11} {r['after']:11} {r['hits']}")
correct = all(r["before"].startswith("ok") and r["after"].startswith("ok") for r in ROWS) and all(
    (r["during"] == "gave up" and r["hits"] == 1) if r["failure"] == "spendcap" else r["during"].startswith("ok")
    for r in ROWS)
print("drill passed:", correct)

failure         group          before      during      after       hits
ratelimit       transient      ok in 1     ok in 3     ok in 1     3
spendcap        fatal          ok in 1     gave up     ok in 1     1
overloaded      transient      ok in 1     ok in 3     ok in 1     3
stream_error    broken stream  ok in 1     ok in 2     ok in 1     2
disconnect      broken stream  ok in 1     ok in 2     ok in 1     2
truncated_tool  truncated      ok in 1     ok in 2     ok in 1     2
drill passed: True


Read the `during` column ("ok in 3" = succeeded on the third try). Transient errors took 3 tries (two failures, then success). Broken streams
took 2. The truncated tool call took 2, the second with `max_tokens=240`. The spend cap stopped after one
request instead of hammering a provider that will say no until next month. `after` is clean everywhere:
nothing stuck from the failure.

Last, the drill result Scout actually needs from the tool case: complete, parsed arguments.

In [15]:
fault("truncated_tool")
print(scout_call(scout_stream)["tool_args"])

{'query': 'Perseverance landing date'}


## At work
- **Overload and outages are normal events.** Anthropic's 529 `overloaded_error` can occur "when the API
  experiences high traffic across all users", so teams keep a second model or provider ready and switch when
  errors pile up. → lesson 57
- **Mid-stream errors are handled by hand.** Errors after a 200 bypass SDK retries; production code
  checks for the final event, discards or resumes partial output, and logs every attempt with its
  failure group so a spike shows up on a dashboard. → lessons 36 and 49
- **Retrying a whole stream re-bills it.** The tokens of a broken attempt were generated and paid for;
  a provider having a bad day can multiply spend, which is one reason for hard cost caps. → lesson 54
- **A retried tool call can run twice.** Retrying the model is safe; re-running a tool that writes or
  sends is not without an idempotency key. → lesson 39

## What just happened
- A stdlib stub spoke Anthropic's streaming format and failed on command in six ways.
- The naive client met five different exception types, including an `APIStatusError` with status 200
  and an `httpx2.RemoteProtocolError` that is not an `anthropic` exception.
- The SDK's own retries fixed the 529 but not the error that arrived mid-stream.
- `classify` sorted every outcome into transient, fatal, broken stream or truncated.
- `scout_call` acted on the group, and the before/during/after table showed every case recovered,
  except the spend cap, which stopped after one request, as it should.

## Common mistakes
**1. Catching only the SDK's exceptions around a stream.** A dropped connection raises the HTTP
library's error, so this `except` never runs.

In [16]:
fault("disconnect", times=1)
try:
    scout_stream()
except anthropic.APIError as e:
    print("handled:", e)

RemoteProtocolError: peer closed connection without sending complete message body (received 651 bytes, expected 1151)

Fix: also catch the transport errors underneath, as `classify` does, and treat them as a broken stream.

In [17]:
fault("disconnect", times=1)
try:
    scout_stream()
except (anthropic.APIError, httpx2.TransportError) as e:
    print("handled:", classify(exc=e))

handled: ('broken stream', 'connection lost: RemoteProtocolError')


**2. Parsing tool arguments without checking the stop reason.** The stream finished cleanly, but
`max_tokens` cut the JSON in half.

In [18]:
fault("truncated_tool")
out = scout_stream()
json.loads(out["tool_json"])

JSONDecodeError: Unterminated string starting at: line 1 column 11 (char 10)

Fix: classify first. `truncated` means "ask again with more room", not "parse what you got".

In [19]:
print(classify(out=out), "|", repr(out["tool_json"]))

('truncated', 'tool JSON cut off by max_tokens') | '{"query": "Perseverance '


## Try it
When the provider keeps failing, the user should get a polite message, not a stack trace. But a spend
cap needs a human, so it must still raise. Write `scout_answer(stream_fn)`: return `scout_call`'s result;
if it gives up on anything except a fatal error, return
`{"text": "Scout is busy right now. Please try again in a minute.", "degraded": True}`.
Test it with the two fakes below (no server needed): `flapping` should come back degraded, `capped`
should raise `GiveUp`.

In [20]:
def flapping(max_tokens):
    raise httpx2.RemoteProtocolError("peer closed connection without sending complete message body")

def capped(max_tokens):
    body = {"error": {"type": "rate_limit_error", "details": {"error_code": "enforced_spend_limit_reached"}}}
    raise anthropic.RateLimitError("spend cap", body=body,
                                   response=httpx2.Response(429, request=httpx2.Request("POST", BASE)))

# your code here

Finally, stop the stub server.

In [21]:
server.terminate()
server.wait(timeout=5)
print("server stopped, exit code", server.returncode)

server stopped, exit code -15


## Key terms
- **failure taxonomy** — sorting every failure into a few groups (transient, fatal, broken stream, truncated), each with one correct action.
- **529 overloaded** — Anthropic's `overloaded_error`: the API as a whole is too busy; temporary, retry with backoff.
- **mid-stream failure** — an error after the server already answered 200 and started streaming: an `event: error`, a dropped connection, or a stream with no final event. SDK retries do not cover it.
- **broken stream** — a streamed reply that did not reach its final `message_stop`; its partial text is incomplete and is thrown away before retrying.
- **truncated tool JSON** — tool arguments cut off because `max_tokens` ran out (`stop_reason: max_tokens`); retry with a larger budget instead of parsing.
- **chaos drill** — breaking a dependency on purpose and checking the system before, during and after the failure.